# 01 — CIC-IDS2017 Exploratory Data Analysis
Loads the preprocessed data and explores class distribution, feature stats, correlation, and imbalance.

In [ ]:
import sys, pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sys.path.insert(0, str(Path().resolve().parent))
from src.configs.paths import PREPROCESSED_DIR, ARTIFACTS_DIR

with open(PREPROCESSED_DIR / 'label_encoder.pkl', 'rb') as f:
    le = pickle.load(f)
with open(PREPROCESSED_DIR / 'feature_cols.pkl', 'rb') as f:
    feature_cols = pickle.load(f)

data = np.load(PREPROCESSED_DIR / 'test_set.npz') if (PREPROCESSED_DIR / 'test_set.npz').exists() else np.load(ARTIFACTS_DIR / 'data' / 'client_0000.npz')
keys = list(data.keys())
X = data[keys[0]]
y = data[keys[1]]

df = pd.DataFrame(X, columns=feature_cols[:X.shape[1]])
df['label'] = y
print(f'Shape: {df.shape}  |  Classes: {len(np.unique(y))}')
df.head()

In [ ]:
# Class distribution
class_names = le.classes_
counts = pd.Series(y).value_counts().sort_index()

fig, ax = plt.subplots(figsize=(14, 5))
bars = ax.bar(range(len(counts)), counts.values, color='steelblue', edgecolor='white')
ax.set_xticks(range(len(counts)))
ax.set_xticklabels([class_names[i] if i < len(class_names) else str(i) for i in counts.index], rotation=60, ha='right', fontsize=8)
ax.set_ylabel('Sample count')
ax.set_title('CIC-IDS2017 — Class Distribution')
ax.set_yscale('log')
plt.tight_layout()
plt.savefig(ARTIFACTS_DIR / 'plots' / 'eda_class_dist.png', dpi=150)
plt.show()

In [ ]:
# Summary statistics
print('Summary statistics (first 10 features):')
df.drop(columns='label').iloc[:, :10].describe().round(3)

In [ ]:
# Feature correlation heatmap
corr = df.drop(columns='label').iloc[:, :20].corr()
fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(corr, cmap='coolwarm', center=0, ax=ax, linewidths=0.3, annot=False)
ax.set_title('Feature Correlation Matrix (first 20 features)')
plt.tight_layout()
plt.savefig(ARTIFACTS_DIR / 'plots' / 'eda_correlation.png', dpi=150)
plt.show()

In [ ]:
# Class imbalance ratio (most common vs rarest)
imbalance = counts.max() / counts.min()
print(f'Imbalance ratio (max/min): {imbalance:.1f}x')
print(f'Most common: {class_names[counts.idxmax()]} ({counts.max():,} samples)')
print(f'Rarest:      {class_names[counts.idxmin()]} ({counts.min():,} samples)')